[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# find and find_one


## What you will be able to do

Read documents: `find_one` for one, `find` for a cursor, and say what a cursor is and why it can
only be walked once. Keep results small with a projection, and say why you cannot mix including and
excluding fields in the same one. Count two ways and know which is exact and which is instant. And
page through a collection without using `skip`, having seen what `skip` makes the server do.


## The idea

### The problem

`find` does not return your documents. It returns a cursor, which is a promise to fetch them in
batches when you ask. That makes reading a million documents possible without a million documents in
memory, and it makes a cursor behave unlike every other container in Python: iterate it twice and
the second pass is empty, with no error.

### What a cursor is

A handle on a query the server is holding open. PyMongo asks for the first batch when you start
iterating, and for more as you exhaust each one. When the documents run out, the cursor is done and
cannot be rewound.

### Why options must come first

`sort`, `limit`, `skip` and `projection` are parts of the query, not operations on results. Once the
first batch has been fetched the query has been sent, so changing it is impossible, and PyMongo says
so rather than silently ignoring you.

### Where this shows up

Everywhere you read. The single-pass rule catches people who pass a cursor to two functions, and the
projection rule catches people who try to say "everything except this" while also saying "this".

### What this notebook covers

`find_one` and `find`. Cursors: one pass, and how to keep the documents if you need them twice.
Projections, including and excluding. `sort`, `limit`, `skip`. Counting, exactly and approximately.
Then the four failures, one of which is the pagination everybody writes first.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import pymongo

client = pymongo.MongoClient("mongodb://127.0.0.1:27017/shop", tz_aware=True)
shop = client.get_default_database()

laptops = shop.products.find({"kind": "laptop"})

print("first pass: ", sum(1 for _ in laptops))
print("second pass:", sum(1 for _ in laptops), "<- and no error at all")

print("find_one gives a", type(shop.products.find_one({"kind": "laptop"})).__name__)
print("find gives a", type(shop.products.find()).__name__, "which is not a list")
client.close()
```

```
first pass:  100
second pass: 0 <- and no error at all
find_one gives a dict
find gives a Cursor which is not a list
```

A hundred documents, then none. The cursor was not emptied by a bug: it was walked to the end, and a
cursor that has been walked to the end has nothing left. Every list-like habit you have will mislead
you here exactly once.


## Setup

Seven imports, MongoDB, the boot cell, and two helpers.

- `pymongo` is the driver, and `time` measures the two ways of counting
- `subprocess` and `os` install and start the server, `sys` names this Python
- `random` seeds the data the same way every run, with `version` and `PackageNotFoundError`

`examined` asks the server, through `explain`, how many index keys and documents it actually had to
look at to answer a query. That number is the only honest way to compare two ways of writing the
same read, because a timing over five hundred documents on a local socket measures nothing.

`failed` prints the message out of an `OperationFailure` rather than the exception itself, which
carries a cluster timestamp and a signature that change every time you run it.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def failed(error):
    """An OperationFailure's real message. Printing the exception whole would include a cluster
    time and a signature, which are different on every run and are never the point."""
    return f"{type(error).__name__}: {error.details.get('errmsg', error)}"


def examined(cursor):
    """What the server actually had to look at, which is the only honest measure of a query."""
    stats = cursor.explain()["executionStats"]
    return {"index keys": stats["totalKeysExamined"],
            "documents": stats["totalDocsExamined"],
            "returned": stats["nReturned"]}


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### One document

`find_one` takes the same filter as `find` and gives you a document or `None`:


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

print("by _id:   ", shop.products.find_one({"_id": 3})["name"])
print("by a field:", shop.products.find_one({"kind": "monitor"})["name"])
print("no filter: ", shop.products.find_one()["name"], "<- whichever one it finds first")
print("no match:  ", shop.products.find_one({"kind": "submarine"}))


by _id:    Dalgo mouse 3
by a field: Belden monitor 1
no filter:  Aster laptop 0 <- whichever one it finds first
no match:   None


`None` rather than an exception, which means `find_one(...)["name"]` is a `TypeError` waiting for
the first request that does not match. Check it, or use `find_one(...) or {}` when a default is
genuinely fine.

"Whichever one it finds first" is worth taking literally: with no sort, the order is whatever the
storage engine finds convenient. It is stable here and you should not rely on it.

### A cursor, and the one pass

`find` returns a cursor, and nothing has been fetched yet:


In [3]:
cursor = shop.products.find({"kind": "laptop"})
print("before iterating:", type(cursor).__name__, "| alive:", cursor.alive)

first = next(cursor)
print("after one document:", first["name"])

rest = sum(1 for _ in cursor)
print("the rest:", rest, "| alive now:", cursor.alive)
print("and starting again:", sum(1 for _ in cursor))


before iterating: Cursor | alive: True
after one document: Aster laptop 0
the rest: 99 | alive now: False
and starting again: 0


If you need the documents twice, keep them:


In [4]:
laptops = list(shop.products.find({"kind": "laptop"}))              # fetched, and now a list

print("in memory:", len(laptops))
print("walk it again:", len(laptops))
print("and again:", sum(1 for product in laptops if product["stock"] == 0), "out of stock")


in memory: 100
walk it again: 100
and again: 0 out of stock


`list(cursor)` is the right answer whenever the result is small and you need it more than once. It
is the wrong answer for a large collection, which is the whole reason cursors exist, and
**The Aggregation Pipeline** is where that starts to matter.

### Projections

Ask for less and less crosses the network:


In [5]:
whole = shop.products.find_one({"_id": 0})
names = shop.products.find_one({"_id": 0}, {"name": 1, "price": 1})

print("every field: ", sorted(whole))
print("two of them: ", sorted(names), "<- _id came along uninvited")
print("without it:  ", sorted(shop.products.find_one({"_id": 0}, {"name": 1, "_id": 0})))


every field:  ['_id', 'kind', 'maker', 'name', 'price', 'size', 'sku', 'stock', 'tags']
two of them:  ['_id', 'name', 'price'] <- _id came along uninvited
without it:   ['name']


`_id` is included unless you exclude it explicitly. That is the one exception to the rule below, and
it exists because `_id` is the only field a document is guaranteed to have.

Otherwise a projection is either a list of fields to keep or a list of fields to drop, never both:


In [6]:
print("keeping: ", sorted(shop.products.find_one({}, {"name": 1, "kind": 1})))
print("dropping:", sorted(shop.products.find_one({}, {"tags": 0, "size": 0, "sku": 0})))
print()
print("a projection into a subdocument works too:",
      shop.products.find_one({"_id": 0}, {"size.w": 1, "_id": 0}))


keeping:  ['_id', 'kind', 'name']
dropping: ['_id', 'kind', 'maker', 'name', 'price', 'stock']

a projection into a subdocument works too: {'size': {'w': 21}}


### sort, limit and skip

These are parts of the query, and they are applied by the server in a fixed order: sort, then skip,
then limit:


In [7]:
cheapest = shop.products.find({"kind": "laptop"}, {"name": 1, "price": 1, "_id": 0}) \
                        .sort("price").limit(3)

for product in cheapest:
    print(f"  {product['price']:8.2f}  {product['name']}")

print()
print("sort takes a direction too:",
      shop.products.find({}, {"price": 1, "_id": 0}).sort("price", -1).limit(1)[0])


     58.26  Dalgo laptop 15
     70.68  Dalgo laptop 375
     90.11  Corvid laptop 390

sort takes a direction too: {'price': 1999.73}


`sort("price")` ascends and `sort("price", -1)` descends. For more than one key, pass a list of
pairs, `sort([("kind", 1), ("price", -1)])`, and the order of that list is the order the server
sorts in, which **Indexes** shows is also the order your index must be in.

### Counting

Two methods, and they are not alternatives:


In [8]:
start = time.perf_counter()
exact = shop.products.count_documents({})
exact_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
quick = shop.products.estimated_document_count()
quick_ms = (time.perf_counter() - start) * 1000

print("count_documents({}):       ", exact, "| it ran an aggregation over the collection")
print("estimated_document_count():", quick, "| it read one number out of the metadata")
print("both right here, and only one of them can take a filter")
print("with a filter:", shop.products.count_documents({"kind": "laptop"}))


count_documents({}):        500 | it ran an aggregation over the collection
estimated_document_count(): 500 | it read one number out of the metadata
both right here, and only one of them can take a filter
with a filter: 100


The timings are not printed because over five hundred documents on a local socket they are noise.
The difference is structural: `count_documents` is exact and gets slower as the collection grows,
`estimated_document_count` is constant time and can be wrong after an unclean shutdown.

Use the estimate for a dashboard, the exact count for anything a person will act on, and neither in
a loop.

### What skip costs

The server has to walk past everything it skips. `explain` shows it:


In [9]:
print("  skip  index keys examined")
for skip in (0, 100, 300, 490):
    page = shop.products.find().sort("_id").skip(skip).limit(10)
    print(f"  {skip:4}  {examined(page)['index keys']}")


  skip  index keys examined
     0  10
   100  110
   300  310
   490  500


Ten, then a hundred and ten, then three hundred and ten. The tenth page costs ten times the first,
and the thousandth page costs a thousand times. `limit(10)` bounds what comes back, not what the
server reads.

This is where MongoDB differs from the page-number pagination of **APIs and JSON**: an API can offer
page numbers cheaply only if something underneath can find page nine hundred without counting to it.

### Pagination that stays the same price, finished

Remember where you got to and ask for what is after it:


In [10]:
def page_after(shop, last_id=None, size=10):
    """One page, and the id to pass in next time. Costs the same on page 1000 as on page 1."""
    query = {} if last_id is None else {"_id": {"$gt": last_id}}
    cursor = shop.products.find(query, {"name": 1}).sort("_id").limit(size)
    rows = list(cursor)                                             # list, because we walk it twice
    return rows, (rows[-1]["_id"] if rows else last_id)


cursor_id, seen = None, 0
print("  page  index keys examined  last id")
for number in range(4):
    query = {} if cursor_id is None else {"_id": {"$gt": cursor_id}}
    cost = examined(shop.products.find(query, {"name": 1}).sort("_id").limit(10))
    rows, cursor_id = page_after(shop, cursor_id)
    seen += len(rows)
    print(f"  {number + 1:4}  {cost['index keys']:19}  {cursor_id}")

print()
print("rows seen:", seen, "and every page cost the same")


  page  index keys examined  last id
     1                   10  9
     2                   10  19
     3                   10  29
     4                   10  39

rows seen: 40 and every page cost the same


Ten keys examined per page, forever, because `{"_id": {"$gt": last}}` is a position in the index
rather than a count from the beginning.

The trade is that you cannot jump to page nine hundred, only forward one page at a time. That is
usually what an interface needs anyway, and when it is not, the honest answer is that page numbers
over a large collection are expensive however you get them.

### Where each part came from

| In `page_after` | What it relies on | The section that showed it |
|---|---|---|
| `{"_id": {"$gt": last_id}}` | a position rather than an offset | What skip costs |
| `.sort("_id")` | the index every collection already has | sort, limit and skip |
| `.limit(size)` | bounding what comes back | sort, limit and skip |
| `{"name": 1}` | a projection keeping the result small | Projections |
| `list(cursor)` | a cursor walked once, kept for two uses | A cursor, and the one pass |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/04-find-and-find-one-solutions.ipynb).

**1.** Find one product by `_id` and one that does not exist, and print both.


In [11]:
# your code here


**2.** Walk a cursor twice and show what the second pass gives you.


In [12]:
# your code here


**3.** Ask for two fields and no `_id`.


In [13]:
# your code here


**4.** Print the three most expensive products.


In [14]:
# your code here


**5.** Count the monitors two ways.


In [15]:
# your code here


**6.** Show that `skip(200)` makes the server examine more than `skip(0)`.


In [16]:
# your code here


## Common errors

### pymongo.errors.InvalidOperation: cannot set options after executing query


In [17]:
started = shop.products.find({"kind": "laptop"})
next(started)                                                       # the query has now been sent

started.sort("price")


InvalidOperation: cannot set options after executing query

`sort` is not something you do to results. It is part of the query, and the query left for the
server the moment you asked for the first document.

Build the whole query before you touch it, which is what the chained form does naturally:


In [18]:
ready = shop.products.find({"kind": "laptop"}, {"name": 1, "price": 1, "_id": 0}) \
                     .sort("price", -1).limit(2)

for product in ready:
    print(" ", product)


  {'name': 'Belden laptop 465', 'price': 1956.4}
  {'name': 'Aster laptop 20', 'price': 1933.27}


### pymongo.errors.OperationFailure: Cannot do exclusion on field in inclusion projection


In [19]:
try:
    shop.products.find_one({}, {"name": 1, "maker": 0})
except pymongo.errors.OperationFailure as error:
    print(failed(error))


OperationFailure: Cannot do exclusion on field maker in inclusion projection


A projection answers one question: which fields do I want, or which fields do I not want. Asking
both is not a harder question, it is an incoherent one, so the server refuses.

`_id` is the one field that may appear as a `0` in an inclusion projection, because it is included
by default and turning it off is the only way to say otherwise:


In [20]:
print("allowed:", sorted(shop.products.find_one({}, {"name": 1, "_id": 0})))
print("allowed:", sorted(shop.products.find_one({}, {"maker": 0, "tags": 0})))


allowed: ['name']
allowed: ['_id', 'kind', 'name', 'price', 'size', 'sku', 'stock']


### No error: the cursor that was already spent


In [21]:
cursor = shop.products.find({"kind": "keyboard"})

total = sum(product["price"] for product in cursor)
count = sum(1 for _ in cursor)                                      # the same cursor, again

print("total:", round(total, 2))
print("count:", count)
print("average:", "cannot be computed, and nothing said so")


total: 97205.07
count: 0
average: cannot be computed, and nothing said so


This is the shape the single-pass rule takes in real code: two loops over one cursor, in two
functions, written months apart. The second one gets nothing, the division by zero happens
somewhere else, and the cursor is long out of scope by the time anybody looks.

Fetch once and keep it, or run the query twice on purpose:


In [22]:
keyboards = list(shop.products.find({"kind": "keyboard"}))

total = sum(product["price"] for product in keyboards)
count = len(keyboards)
print("total:", round(total, 2), "| count:", count, "| average:", round(total / count, 2))


total: 97205.07 | count: 100 | average: 972.05


### No error: find_one on nothing


In [23]:
missing = shop.products.find_one({"kind": "submarine"})
print("what came back:", missing)

try:
    print(missing["name"])
except TypeError as error:
    print("TypeError:", error)


what came back: None
TypeError: 'NoneType' object is not subscriptable


The failure is one line later than the cause, which is what makes it annoying: the query was fine,
the filter was fine, and there simply is no such product.

Say what should happen when there is nothing, at the point where you ask:


In [24]:
def name_of(shop, kind):
    found = shop.products.find_one({"kind": kind}, {"name": 1, "_id": 0})
    return found["name"] if found else f"no {kind} in the catalog"


print(name_of(shop, "monitor"))
print(name_of(shop, "submarine"))


Belden monitor 1
no submarine in the catalog


In [25]:
client.close()
print("closed")


closed


## Recap

- `find_one` gives a document or `None`. `find` gives a `Cursor`, which is not a list and has
  fetched nothing yet.
- A cursor can be walked once. The second pass is empty and raises nothing, so keep the documents
  with `list(cursor)` when you need them twice.
- `sort`, `limit`, `skip` and the projection are part of the query. Setting one after iteration has
  started raises `InvalidOperation`.
- A projection either includes fields or excludes them, never both, and `_id` is the single
  exception because it is included unless you say otherwise.
- `count_documents(filter)` is exact and scans. `estimated_document_count()` reads metadata, is
  instant, takes no filter, and can be wrong after an unclean shutdown.
- `skip(n)` makes the server walk past `n` entries, so the cost of a page grows with its number.
  Paging by `{"_id": {"$gt": last}}` costs the same on every page.


## What is next

**Query Operators** is the filter itself: why `{"size": {"w": 1}}` matches nothing while
`{"size.w": 1}` matches every document where `w` is 1, and why two conditions on one array can both
be true of a document that has no element satisfying either.


---

&#8592; **Previous:** [BSON Types](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/03-bson-types.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
